# ETHZ Food-101: HBCC vs ResNet-18 (Kaggle)

Notebook độc lập để chạy trên Kaggle, tải split chính thức ETHZ Food-101 bằng `torchvision.datasets.Food101`. Mọi ảnh được resize về **224×224**, không dùng data augmentation.

In [ ]:
# Cấu hình — chỉnh tại đây trước khi Run All
SEED = 42
DATA_ROOT = '/kaggle/working/torchvision_data'
VAL_FRACTION = 0.10      # stratified: lấy 10% train chính thức làm validation
IMAGE_SIZE = 224
BATCH_SIZE = 32          # giảm xuống 16/8 nếu GPU thiếu VRAM
NUM_WORKERS = 2
EPOCHS = 20
LR = 3e-4
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.1
MODELS_TO_RUN = ['hbcc', 'resnet18']  # ví dụ: ['hbcc']
RESNET18_PRETRAINED = True  # True: fine-tune ImageNet; False: so sánh khởi tạo ngẫu nhiên công bằng
USE_AMP = True
OUTPUT_DIR = '/kaggle/working/food101_hbcc_resnet18'


In [ ]:
import os, random, time, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import transforms, models, datasets
from torchvision.transforms import InterpolationMode

def seed_everything(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True
seed_everything(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
print('device:', device, '| torch:', torch.__version__)


## Tải Food-101 bằng PyTorch
`torchvision.datasets.Food101` cung cấp 750 ảnh train và 250 ảnh test cho mỗi lớp. Notebook tách stratified 10% từ train thành validation, nên mỗi lớp có 675 train / 75 validation / 250 test. Test được giữ kín đến cuối. Trên Kaggle cần bật **Internet** cho lần tải đầu tiên.

In [ ]:
Path(DATA_ROOT).mkdir(parents=True, exist_ok=True)
print('Torchvision data root:', DATA_ROOT)


In [ ]:
# Cùng một transform xác định cho train và test: hoàn toàn không augmentation.
data_tf = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE), interpolation=InterpolationMode.BILINEAR),
    transforms.ToTensor(), transforms.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225])
])

print('Đang tải/kiểm tra Food-101 từ PyTorch...')
full_train_dataset = datasets.Food101(root=DATA_ROOT, split='train', download=True, transform=data_tf)
test_dataset = datasets.Food101(root=DATA_ROOT, split='test', download=True, transform=data_tf)
classes = list(full_train_dataset.classes)
class_to_idx = {name: i for i, name in enumerate(classes)}
assert classes == list(test_dataset.classes)
assert len(classes) == 101 and len(full_train_dataset) == 75750 and len(test_dataset) == 25250

# Stratified split xác định: 675 train + 75 validation cho mỗi lớp.
all_labels = np.array([class_to_idx[label] for label in full_train_dataset._labels], dtype=np.int64)
split_rng = np.random.default_rng(SEED)
train_indices, val_indices = [], []
for class_idx in range(len(classes)):
    class_indices = np.flatnonzero(all_labels == class_idx)
    split_rng.shuffle(class_indices)
    n_val = int(round(len(class_indices) * VAL_FRACTION))
    val_indices.extend(class_indices[:n_val].tolist())
    train_indices.extend(class_indices[n_val:].tolist())
train_dataset = Subset(full_train_dataset, train_indices)
val_dataset = Subset(full_train_dataset, val_indices)
assert len(train_dataset) == 68175 and len(val_dataset) == 7575
assert set(train_indices).isdisjoint(val_indices) and len(train_indices)+len(val_indices) == len(full_train_dataset)
print(f'{len(classes)} classes | train={len(train_dataset):,} | val={len(val_dataset):,} | test={len(test_dataset):,}')

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=NUM_WORKERS>0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=NUM_WORKERS>0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=NUM_WORKERS>0)
x, y = next(iter(train_loader))
assert x.shape[1:] == (3, IMAGE_SIZE, IMAGE_SIZE)
print('Batch:', x.shape, '| targets:', y.shape, '| no augmentation')


## HBCC (self-contained) và ResNet-18
HBCC-Medium đã được đối chiếu với mã trong repo: dims 64/96/192/288, depths 1/1/2/2, heads 2/3/4/4, cosine hard assignment (không hard-ST), proposal 2×2, folds 4/2/1/1, nhánh LBP/DWConv/cluster/DWConv, local ratios 0.5/0.5/0/0.25 và channel shuffle ở các stage hybrid. Điều chỉnh cho input 224 là stem 7×7 stride 4 để tạo feature maps 56/28/14/7.

In [ ]:
class DropPath(nn.Module):
    def __init__(self, p=0.): super().__init__(); self.p=p
    def forward(self, x):
        if self.p == 0 or not self.training: return x
        mask = x.new_empty((x.shape[0],)+(1,)*(x.ndim-1)).bernoulli_(1-self.p)
        return x * mask / (1-self.p)

class Coord(nn.Module):
    def forward(self, x):
        b,_,h,w=x.shape; yy=torch.linspace(-.5,.5,h,device=x.device,dtype=x.dtype); xx=torch.linspace(-.5,.5,w,device=x.device,dtype=x.dtype)
        gy,gx=torch.meshgrid(yy,xx,indexing='ij'); return torch.cat([x, gx.expand(b,1,-1,-1), gy.expand(b,1,-1,-1)],1)

def channel_shuffle(x, groups=2):
    b,c,h,w=x.shape
    if groups <= 1 or c % groups: return x
    return x.reshape(b,groups,c//groups,h,w).transpose(1,2).contiguous().reshape(b,c,h,w)

class Local(nn.Module):
    def __init__(self, c, kind):
        super().__init__(); self.kind=kind
        if kind=='dw': self.net=nn.Sequential(nn.Conv2d(c,c,3,1,1,groups=c,bias=False),nn.BatchNorm2d(c),nn.GELU(),nn.Conv2d(c,c,1),nn.BatchNorm2d(c),nn.GELU())
        elif kind=='lbp':
            self.dw=nn.Conv2d(c,c*8,3,1,1,groups=c,bias=False); self.dw.weight.requires_grad=False
            with torch.no_grad():
                f=torch.zeros(8,1,3,3); pts=[(0,0),(0,1),(0,2),(1,2),(2,2),(2,1),(2,0),(1,0)]
                for i,(r,col) in enumerate(pts): f[i,0,r,col]=1; f[i,0,1,1]=-1
                self.dw.weight.copy_(f.repeat(c,1,1,1))
            self.net=nn.Sequential(nn.BatchNorm2d(c*8),nn.GELU(),nn.Conv2d(c*8,c,1),nn.BatchNorm2d(c),nn.GELU())
        else: self.net=nn.Identity()
    def forward(self,x): return self.net(self.dw(x)) if self.kind=='lbp' else self.net(x)

class ContextCluster(nn.Module):
    def __init__(self, dim, heads=2, head_dim=16, proposal=(2,2), fold=(1,1)):
        super().__init__(); self.h,self.d,self.proposal,self.fold=heads,head_dim,proposal,fold
        self.f=nn.Conv2d(dim,heads*head_dim,1); self.v=nn.Conv2d(dim,heads*head_dim,1); self.proj=nn.Conv2d(heads*head_dim,dim,1)
        self.alpha=nn.Parameter(torch.ones(1)); self.beta=nn.Parameter(torch.zeros(1))
    def forward(self,x):
        b,_,H,W=x.shape; fh,fw=self.fold; assert H%fh==0 and W%fw==0
        def part(z):
            z=z.reshape(b,self.h,self.d,H,W).reshape(b,self.h,self.d,fh,H//fh,fw,W//fw).permute(0,1,3,5,2,4,6)
            return z.reshape(-1,self.d,H//fh,W//fw)
        feat,val=part(self.f(x)),part(self.v(x)); bh,c,h,w=feat.shape
        cen=F.adaptive_avg_pool2d(feat,self.proposal).flatten(2).transpose(1,2); vcen=F.adaptive_avg_pool2d(val,self.proposal).flatten(2).transpose(1,2)
        pts=feat.flatten(2).transpose(1,2); values=val.flatten(2).transpose(1,2)
        logits=self.beta+self.alpha*torch.matmul(F.normalize(cen,dim=-1),F.normalize(pts,dim=-1).transpose(-2,-1))
        max_idx=logits.max(dim=1,keepdim=True).indices
        assign=torch.zeros_like(logits); assign.scatter_(1,max_idx,1.0)  # hard assignment thuần, không STE/softmax
        sim=torch.sigmoid(logits)*assign; agg=((values[:,None]*sim[...,None]).sum(2)+vcen)/(sim.sum(-1,keepdim=True)+1)
        out=(agg[:,:,None]*sim[...,None]).sum(1).transpose(1,2).reshape(b,self.h,fh,fw,c,h,w).permute(0,1,4,2,5,3,6).reshape(b,self.h*c,H,W)
        return self.proj(out)

class Mlp(nn.Module):
    # Khớp MLP channel-last của implementation HBCC trong repo.
    def __init__(self, channels, ratio=3.0, drop=0.0):
        super().__init__(); hidden=int(channels*ratio)
        self.fc1=nn.Linear(channels,hidden); self.act=nn.GELU(); self.drop=nn.Dropout(drop); self.fc2=nn.Linear(hidden,channels)
    def forward(self,x):
        x=x.permute(0,2,3,1); x=self.drop(self.act(self.fc1(x))); x=self.drop(self.fc2(x))
        return x.permute(0,3,1,2).contiguous()

class HBCCBlock(nn.Module):
    def __init__(self, dim, heads, fold, local_ratio, local_kind, drop_path, shuffle=False):
        super().__init__(); self.n1=nn.BatchNorm2d(dim); self.n2=nn.BatchNorm2d(dim); ld=int(round(dim*local_ratio)); self.cd=dim-ld; self.shuffle=shuffle
        self.cluster=ContextCluster(self.cd,heads,16,(2,2),fold) if self.cd else nn.Identity(); self.local=Local(ld,local_kind) if ld else nn.Identity(); self.fuse=nn.Conv2d(dim,dim,1) if ld and self.cd else nn.Identity()
        self.mlp=Mlp(dim,ratio=3.0,drop=0.0); self.dp=DropPath(drop_path); self.g1=nn.Parameter(1e-5*torch.ones(dim)); self.g2=nn.Parameter(1e-5*torch.ones(dim))
    def forward(self,x):
        z=self.n1(x)
        if self.cd == z.shape[1]: z=self.cluster(z)
        elif self.cd == 0: z=self.local(z)
        else:
            z=torch.cat([self.cluster(z[:,:self.cd]),self.local(z[:,self.cd:])],1)
            if self.shuffle: z=channel_shuffle(z,2)
        z=self.fuse(z); x=x+self.dp(self.g1[None,:,None,None]*z)
        return x+self.dp(self.g2[None,:,None,None]*self.mlp(self.n2(x)))

class HBCCNet(nn.Module):
    def __init__(self, n_classes=101, dims=(64,96,192,288), depths=(1,1,2,2)):
        super().__init__(); self.coord=Coord(); self.stem=nn.Sequential(nn.Conv2d(5,dims[0],7,4,3,bias=False),nn.BatchNorm2d(dims[0]),nn.GELU())
        folds=[(4,4),(2,2),(1,1),(1,1)]; heads=[2,3,4,4]; ratios=[.5,.5,0,.25]; kinds=['lbp','dw','id','dw']; shuffles=[True,True,False,True]; modes=[]
        for i,d in enumerate(dims): modes.append(nn.Sequential(*[HBCCBlock(d,heads[i],folds[i],ratios[i],kinds[i],.08*(sum(depths[:i])+j)/(sum(depths)-1),shuffles[i]) for j in range(depths[i])]))
        self.stages=nn.ModuleList(modes); self.down=nn.ModuleList([nn.Sequential(nn.Conv2d(dims[i],dims[i+1],3,2,1,bias=False),nn.BatchNorm2d(dims[i+1]),nn.GELU()) for i in range(3)])
        self.norm=nn.BatchNorm2d(dims[-1]); self.head=nn.Linear(dims[-1],n_classes)
    def forward_intermediates(self,x):
        x=self.stem(self.coord(x))
        features=[]
        for i,s in enumerate(self.stages):
            x=s(x)
            if i == len(self.stages)-1: x=self.norm(x)
            features.append(x)
            if i<3: x=self.down[i](x)
        return tuple(features)
    def forward(self,x):
        x=self.forward_intermediates(x)[-1]
        return self.head(x.mean((2,3)))

def build_model(name):
    if name == 'hbcc': return HBCCNet(len(classes))
    weights = models.ResNet18_Weights.IMAGENET1K_V1 if RESNET18_PRETRAINED else None
    model = models.resnet18(weights=weights); model.fc = nn.Linear(model.fc.in_features, len(classes)); return model

for name in MODELS_TO_RUN:
    m=build_model(name).eval()
    with torch.inference_mode(): output=m(torch.randn(2,3,224,224))
    assert output.shape == (2,101)
    if name == 'hbcc':
        with torch.inference_mode(): shapes=[tuple(f.shape) for f in m.forward_intermediates(torch.randn(1,3,224,224))]
        assert [s[-2:] for s in shapes] == [(56,56),(28,28),(14,14),(7,7)], shapes
        print('HBCC stage shapes:', shapes)
    print(name, f'{sum(p.numel() for p in m.parameters()):,} parameters', '| output:', output.shape)
    del m


In [ ]:
@torch.inference_mode()
def evaluate(model, loader):
    model.eval(); total=correct=loss_sum=0; criterion=nn.CrossEntropyLoss()
    for images, targets in loader:
        images,targets=images.to(device,non_blocking=True),targets.to(device,non_blocking=True)
        logits=model(images); loss_sum += criterion(logits,targets).item()*targets.size(0); correct += (logits.argmax(1)==targets).sum().item(); total += targets.size(0)
    return loss_sum/total, 100*correct/total

def train_one_model(name):
    model=build_model(name).to(device); criterion=nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
    optimizer=torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=WEIGHT_DECAY); scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,EPOCHS)
    scaler=torch.amp.GradScaler('cuda', enabled=USE_AMP and device.type=='cuda'); history=[]; best=-float('inf')
    checkpoint_path=Path(OUTPUT_DIR)/f'{name}_best.pt'
    for epoch in range(1,EPOCHS+1):
        model.train(); total=correct=loss_sum=0; start=time.time()
        for images,targets in train_loader:
            images,targets=images.to(device,non_blocking=True),targets.to(device,non_blocking=True); optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast('cuda', enabled=USE_AMP and device.type=='cuda'): logits=model(images); loss=criterion(logits,targets)
            scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
            loss_sum+=loss.item()*targets.size(0); correct+=(logits.detach().argmax(1)==targets).sum().item(); total+=targets.size(0)
        val_loss,val_acc=evaluate(model,val_loader); scheduler.step(); row={'model':name,'epoch':epoch,'train_loss':loss_sum/total,'train_acc':100*correct/total,'val_loss':val_loss,'val_acc':val_acc,'seconds':time.time()-start}; history.append(row)
        print(f"{name} | {epoch:02d}/{EPOCHS} | train {row['train_acc']:.2f}% | val {val_acc:.2f}% | {row['seconds']:.0f}s")
        if val_acc>best:
            best=val_acc
            torch.save({'model':name,'epoch':epoch,'val_acc':val_acc,'class_to_idx':class_to_idx,'state_dict':model.state_dict()},checkpoint_path)
    # Test đúng một lần, sau khi chọn checkpoint tốt nhất hoàn toàn bằng validation.
    checkpoint=torch.load(checkpoint_path,map_location=device)
    model.load_state_dict(checkpoint['state_dict'])
    test_loss,test_acc=evaluate(model,test_loader)
    test_row={'model':name,'best_epoch':checkpoint['epoch'],'best_val_acc':checkpoint['val_acc'],'test_loss':test_loss,'test_acc':test_acc}
    print(f"{name} | FINAL TEST | best epoch {checkpoint['epoch']} | val {checkpoint['val_acc']:.2f}% | test {test_acc:.2f}%")
    return pd.DataFrame(history), test_row

run_outputs=[train_one_model(name) for name in MODELS_TO_RUN]
results=pd.concat([item[0] for item in run_outputs],ignore_index=True)
test_results=pd.DataFrame([item[1] for item in run_outputs])
results.to_csv(Path(OUTPUT_DIR)/'history.csv',index=False)
test_results.to_csv(Path(OUTPUT_DIR)/'test_results.csv',index=False)


In [ ]:
fig,ax=plt.subplots(1,2,figsize=(14,4))
for name,g in results.groupby('model'):
    ax[0].plot(g.epoch,g.train_acc,label=name); ax[1].plot(g.epoch,g.val_acc,label=name)
ax[0].set(title='Train accuracy',xlabel='Epoch',ylabel='%'); ax[1].set(title='Validation accuracy',xlabel='Epoch',ylabel='%')
for a in ax: a.grid(); a.legend()
plt.show()
display(results.loc[results.groupby('model').val_acc.idxmax()].sort_values('val_acc',ascending=False))
display(test_results.sort_values('test_acc',ascending=False))
print('Đã lưu checkpoint và history tại:', OUTPUT_DIR)
